# TTC Subway Delay Analysis

In [ ]:
#| include: false
!pip install geopandas

In [ ]:
#| include: false
!pip install openmeteo-requests

In [ ]:
#| include: false
!pip install requests-cache retry-requests numpy pandas 

#### Imports

In [ ]:
#| include: false
import logging
logging.getLogger().setLevel(logging.ERROR)  # Suppress info/warning messages
import warnings
warnings.filterwarnings("ignore") 
import requests
from io import BytesIO
import pandas as pd
import geopandas as gpd
import time
import openmeteo_requests
import requests_cache
from retry_requests import retry
import os
import difflib
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point
from scipy.stats import ttest_ind
import statsmodels.api as sm
import statsmodels.formula.api as smf

### Import TTC Subway Delay Data from 2024

In [ ]:
#| include: false

delay_df = pd.read_excel("../data/ttc-subway-delay-data-2024.xlsx")

# Display first few rows of the dataset
print(delay_df.head())
print(delay_df.info())


### Import Toronto Neighbourhood Profiles

In [ ]:
#| include: false

# Load the Excel file
file_path = "../data/neighbourhood-profiles-2021-158-model.xlsx"

# Read the first sheet (modify `sheet_name` if necessary)
df = pd.read_excel(file_path, sheet_name=0)

# Set the first column as the index and transpose the DataFrame
df_t = df.set_index("Neighbourhood Name").transpose()

# # Print column names to verify structure
# print(df_t.columns.tolist())

# Extract only the Neighbourhood ID and Population columns
pop_df = df_t[["Neighbourhood Number", "Total - Age groups of the population - 25% sample data"]].copy()

# Rename columns for clarity
pop_df.rename(columns={"Neighbourhood Number": "neighbourhood_id",
                       "Total - Age groups of the population - 25% sample data": "population"}, inplace=True)

# Convert data types
pop_df["neighbourhood_id"] = pd.to_numeric(pop_df["neighbourhood_id"], errors="coerce")
pop_df["population"] = pd.to_numeric(pop_df["population"], errors="coerce")

# Display the cleaned DataFrame
print(pop_df.head())
print(pop_df.info())

### Import Toronto Neighbourhoods Shapefile

In [ ]:
#| include: false

# Set the file path to your shapefile (ensure it's in the same directory or provide the full path)
shapefile_path = "../data/Neighbourhoods - 4326/Neighbourhoods - 4326.shp"  # Adjust filename if necessary

# Load the shapefile using GeoPandas
toronto_gdf = gpd.read_file(shapefile_path)

# Display basic information about the shapefile
print(toronto_gdf.info())

# Display the first few rows
print(toronto_gdf.head())


### Create dictionary of subway station names

In [ ]:
#| include: false

# Define TTC subway stations by line
stations_by_line = {
    'YU': [
        "FINCH STATION", "NORTH YORK CENTRE STATION", "SHEPPARD-YONGE STATION", "YORK MILLS STATION",
        "LAWRENCE STATION", "EGLINTON STATION", "DAVISVILLE STATION", "ST CLAIR STATION",
        "SUMMERHILL STATION", "ROSEDALE STATION", "BLOOR-YONGE STATION", "WELLESLEY STATION",
        "COLLEGE STATION", "DUNDAS STATION", "QUEEN STATION", "KING STATION", "UNION STATION",
        "ST ANDREW STATION", "OSGOODE STATION", "ST PATRICK STATION", "QUEEN'S PARK STATION",
        "MUSEUM STATION", "ST GEORGE STATION", "SPADINA STATION", "DUPONT STATION",
        "ST CLAIR WEST STATION", "EGLINTON WEST STATION", "GLENCAIRN STATION", "LAWRENCE WEST STATION",
        "YORKDALE STATION", "WILSON STATION", "SHEPPARD WEST STATION", "DOWNSVIEW PARK STATION",
        "FINCH WEST STATION", "YORK UNIVERSITY STATION", "PIONEER VILLAGE STATION",
        "HIGHWAY 407 STATION", "VAUGHAN METROPOLITAN CENTRE STATION"
    ],
    'BD': [
        "KIPLING STATION", "ISLINGTON STATION", "ROYAL YORK STATION", "OLD MILL STATION",
        "JANE STATION", "RUNNYMEDE STATION", "HIGH PARK STATION", "KEELE STATION",
        "DUNDAS WEST STATION", "LANSDOWNE STATION", "DUFFERIN STATION", "OSSINGTON STATION",
        "CHRISTIE STATION", "BATHURST STATION", "SPADINA STATION", "ST GEORGE STATION",
        "BAY STATION", "BLOOR-YONGE STATION", "SHERBOURNE STATION", "CASTLE FRANK STATION",
        "BROADVIEW STATION", "CHESTER STATION", "PAPE STATION", "DONLANDS STATION",
        "GREENWOOD STATION", "COXWELL STATION", "WOODBINE STATION", "MAIN STREET STATION",
        "VICTORIA PARK STATION", "WARDEN STATION", "KENNEDY STATION"
    ],
    'SHP': [
        "SHEPPARD-YONGE STATION", "BAYVIEW STATION", "BESSARION STATION",
        "LESLIE STATION", "DON MILLS STATION"
    ]
}


### Retrieve coordinates of subway stations

In [ ]:
#| include: false

def get_station_coordinates(station_name):
    """
    Retrieves the latitude and longitude of a TTC subway station using OpenStreetMap's Nominatim API.
    Handles errors and retries in case of failure.
    """
    base_url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": f"{station_name}, Ontario, Canada",
        "format": "json",
        "limit": 1
    }

    try:
        response = requests.get(base_url, params=params, headers={"User-Agent": "Mozilla/5.0"})
        
        # Check if response is empty or the request failed
        if response.status_code != 200:
            print(f"Error: Received status code {response.status_code} for {station_name}")
            return None, None

        data = response.json()
        
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
        else:
            print(f"Warning: No coordinates found for {station_name}")
            return None, None

    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return None, None
    except ValueError:
        print(f"JSON decoding error for {station_name}")
        return None, None

In [ ]:
#| include: false

# Get all unique station names from stations_by_line
unique_stations = set(station for stations in stations_by_line.values() for station in stations)

# Create an empty list to store station data
station_data = []

# Retrieve coordinates for each unique station
for station in unique_stations:
    lat, lon = get_station_coordinates(station)
    station_data.append({"station": station, "latitude": lat, "longitude": lon})
    print(f"Retrieved: {station} -> ({lat}, {lon})")
    time.sleep(1.5)  # Delay to prevent exceeding API rate limits

# Convert list to a DataFrame
stations_df = pd.DataFrame(station_data)

# Display first few rows
print(stations_df.head())
print(stations_df.info())

### Retrieve Hourly Weather Data from 2024

In [ ]:
#| include: false

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 43.7001,
    "longitude": -79.4163,
    "start_date": "2024-01-01",
    "end_date": "2024-12-31",
    "hourly": ["temperature_2m", "relative_humidity_2m", "apparent_temperature", "precipitation", "rain", "snowfall", "snow_depth", "cloud_cover", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "is_day"],
    "timezone": "America/New_York",
    "temporal_resolution": "native"
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation {response.Elevation()} m asl")
print(f"Timezone {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(2).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(3).ValuesAsNumpy()
hourly_rain = hourly.Variables(4).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(5).ValuesAsNumpy()
hourly_snow_depth = hourly.Variables(6).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(7).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(8).ValuesAsNumpy()
hourly_wind_gusts_10m = hourly.Variables(10).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
    start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
    end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
    freq = pd.Timedelta(seconds = hourly.Interval()),
    inclusive = "left"
)}

hourly_data["temperature"] = hourly_temperature_2m
hourly_data["relative_humidity"] = hourly_relative_humidity_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["snowfall"] = hourly_snowfall
hourly_data["snow_depth"] = hourly_snow_depth
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["wind_speed"] = hourly_wind_speed_10m
hourly_data["wind_gusts"] = hourly_wind_gusts_10m

weather_df = pd.DataFrame(data = hourly_data)
print(weather_df)

print(weather_df.head())
print(weather_df.info())


#### Save stations data and weather data in data folder

In [ ]:
#| include: false

# save dfs retrieved using apis

# Define the path to the data folder
data_folder = "data"

# Create the folder if it doesn't exist
os.makedirs(data_folder, exist_ok=True)

# Save stations_df and weather_df as CSV
stations_df.to_csv(os.path.join(data_folder, "stations_data.csv"), index=False)
weather_df.to_csv(os.path.join(data_folder, "weather_data.csv"), index=False)

print("Files saved successfully in the data folder!")

### Data Cleaning

#### Check delay_df

In [ ]:
#| include: false
print(delay_df.info())

In [ ]:
#| include: false

# Remove a 'Min Gap' and 'Vehicle' column
delay_df = delay_df.drop(columns=["Min Gap"])
delay_df = delay_df.drop(columns=["Vehicle"])

In [ ]:
#| include: false

# Rename columns for clarity
delay_df.rename(columns={"Date": "date", "Time": "time", "Day": "day", "Station": "station", "Code": "code", 
                         "Min Delay": "min_delay", "Bound": "bound", "Line": "line"}, inplace=True)

In [ ]:
#| include: false
# Checking for missing values and duplicates in delay_df
print("\nMissing Values in Delay DataFrame:\n", delay_df.isnull().sum())
print("\nDuplicate Rows in Delay DataFrame:", delay_df.duplicated().sum())

In [ ]:
#| include: false
# Handling missing values in delay data
delay_df = delay_df.dropna(subset=['bound'])  # Drop rows where min_delay is missing
delay_df = delay_df.dropna(subset=['line'])

# Drop duplicate rows in delay_df
delay_df = delay_df.drop_duplicates()

# Checking for missing values and duplicates in delay_df
print("\nMissing Values in Delay DataFrame:\n", delay_df.isnull().sum())
print("\nDuplicate Rows in Delay DataFrame:", delay_df.duplicated().sum())

print("Delay DataFrame Shape:", delay_df.shape)

In [ ]:
#| include: false
print("Unique values in 'day' column:", delay_df['day'].unique())

day_counts = delay_df['day'].value_counts()

print(day_counts)

In [ ]:
#| include: false


# Define the correct order for days
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Convert 'day' column to a categorical type with the specified order
delay_df['day'] = pd.Categorical(delay_df['day'], categories=day_order, ordered=True)

In [ ]:
#| include: false
print("Unique values in 'code' column:", delay_df['code'].unique())

code_counts = delay_df['code'].value_counts()

print(code_counts)

In [ ]:
#| include: false

# Convert 'code' column to category type
delay_df['code'] = delay_df['code'].astype('category')

In [ ]:
#| include: false
print("Unique values in 'bound' column:", delay_df['bound'].unique())

bound_counts = delay_df['bound'].value_counts()

print(bound_counts)

In [ ]:
#| include: false
# Drop rows where 'bound' column is exactly 'B'
delay_df = delay_df.loc[delay_df['bound'] != 'B']

# Display the updated shape of the DataFrame
print("Updated Delay DataFrame Shape:", delay_df.shape)

In [ ]:
#| include: false

# Convert 'bound' column to category type
delay_df['bound'] = delay_df['bound'].astype('category')

In [ ]:
#| include: false
print("Unique values in 'line' column:", delay_df['line'].unique())

line_counts = delay_df['line'].value_counts()

print(line_counts)

In [ ]:
#| include: false
print(delay_df.loc[delay_df['line'] == "YUS"])

In [ ]:
#| include: false
delay_df.loc[delay_df['line'] == 'YUS', 'line'] = 'YU'
print("Delay DataFrame Shape:", delay_df.shape)

In [ ]:
#| include: false
print("Unique values in 'line' column:", delay_df['line'].unique())

line_counts = delay_df['line'].value_counts()

print(line_counts)

In [ ]:
#| include: false

# Convert 'line' column to category type
delay_df['line'] = delay_df['line'].astype('category')

In [ ]:
#| include: false
print("Unique values in 'station' column:", delay_df['station'].unique())

station_counts = delay_df['station'].value_counts()

print(station_counts)

In [ ]:
#| include: false

def clean_station_name(station, line):
    """ Cleans and standardizes station names while keeping 'STATION' and handling edge cases. """

    # Handle 'X TO Y' cases by keeping 'Y'
    if " TO " in station:
        station = station.split(" TO ")[-1]
    elif " - " in station:
        station = station.split(" - ")[-1]

    # Remove extra text like "(APPROA", "(TOWARD", "(LEAVIN"
    station = station.split("(")[0].strip()

    # Ensure "STATION" is present in the cleaned name
    if "STATION" not in station:
        station += " STATION"

    # Fix known typos
    typo_fixes = {
        "BLOOR SATION STATION": "BLOOR STATION",
        "PAPE STATIO STATION": "PAPE STATION",
        "YONGE BD  (ON R STATION": "BLOOR-YONGE STATION",
        "MUSUEM STATION": "MUSEUM STATION",
        "NORTH YORK CTR STATION": "NORTH YORK CENTRE STATION"
    }
    if station in typo_fixes:
        return typo_fixes[station]
    
    # Fix known edge cases where the line name is in the station name
    line_fixes = {
        "YONGE BD STATION": "BLOOR-YONGE STATION",
        "VAUGHAN MC STATION": "VAUGHAN METROPOLITAN CENTRE STATION",
        "SPADINA BD STATION": "SPADINA STATION",
        "SPADINA YUS STATION": "SPADINA STATION",
        "ST GEORGE YUS STATION": "ST GEORGE STATION",
        "ST GEORGE BD STATION": "ST GEORGE STATION",
        "KENNEDY BD STATION": "KENNEDY STATION"
    }
    if station in line_fixes:
        return line_fixes[station]

    # Handle intersections (e.g., "BLOOR-YONGE STATION" and "SHEPPARD-YONGE STATION")
    if "YONGE" in station and "BLOOR" in station:
        return "BLOOR-YONGE STATION"
    if "YONGE" in station and "SHEPPARD" in station:
        return "SHEPPARD-YONGE STATION"

    # Get the correct station list based on the line
    valid_stations = stations_by_line.get(line, [])

    # Find closest match using difflib
    closest_matches = difflib.get_close_matches(station, valid_stations, n=1, cutoff=0.8)

    # Return the best match if found, else return the station with "STATION" added
    return closest_matches[0] if closest_matches else station


In [ ]:
#| include: false
# Apply cleaning function to the 'station' column
delay_df['cleaned_station'] = delay_df.apply(lambda row: clean_station_name(row['station'], row['line']), axis=1)

# Display before/after cleaning
print(delay_df[['station', 'cleaned_station']].head(20))


In [ ]:
#| include: false
# Create a set of all valid station names from stations_by_line
valid_stations = set(station for stations in stations_by_line.values() for station in stations)

# Find non-matching stations
non_matching_stations = delay_df[~delay_df['cleaned_station'].isin(valid_stations)]

# Display the count of affected rows
print(f"Number of rows with non-matching station names: {non_matching_stations.shape[0]}")
print("Sample of non-matching stations:")
print(non_matching_stations[['station', 'cleaned_station']].head(10))  # Display a few examples


In [ ]:
#| include: false
# Remove rows where 'cleaned_station' is not in the valid TTC station list
delay_df = delay_df[delay_df['cleaned_station'].isin(valid_stations)]

# Display the updated shape of the DataFrame
print("Updated Delay DataFrame Shape:", delay_df.shape)


In [ ]:
#| include: false
print("Unique values in 'cleaned_station' column:", delay_df['cleaned_station'].unique())

cleaned_station_counts = delay_df['cleaned_station'].value_counts()

print(cleaned_station_counts)

In [ ]:
#| include: false

# Change 'cleaned_station' column to category type
delay_df['cleaned_station'] = delay_df['cleaned_station'].astype('category')

In [ ]:
#| include: false

# Remove a 'station' column
delay_df = delay_df.drop(columns=["station"])

print(delay_df.info())

In [ ]:
#| include: false
# Summary statistics for Delay Data (formatted)
delay_summary = delay_df.describe().T  # Transpose for better readability
delay_summary.style.set_caption("Delay Data Summary").format("{:.2f}")


In [ ]:
#| include: false
# The maximum value in min_delay is 716 minutes, just under 12 hours. 
# I will check how many delays lasted longer than one hour.

In [ ]:
#| include: false
# Get all rows where min_delay is greater than 10
high_delay_df = delay_df.loc[delay_df['min_delay'] > 30]

print(high_delay_df.shape)

# Display the first few rows
print(high_delay_df.head())


In [ ]:
#| include: false
# Get all rows where min_delay is equalt to 0
no_delay_df = delay_df.loc[delay_df['min_delay'] == 0]

print(no_delay_df.shape)

# Display the first few rows
print(no_delay_df.head())

In [ ]:
#| include: false
# I will focus on delays that lasted for at most one hour, as only 147 delays lasted longer than this. 
# Additionally, these significantly longer delays were most likely due to major incidents, 
# which are outside the scope of this study.

In [ ]:
#| include: false
delay_df = delay_df.loc[delay_df['min_delay'] <= 30]
print("Delay DataFrame Shape:", delay_df.shape)

In [ ]:
#| include: false
delay_df = delay_df.loc[delay_df['min_delay'] > 0]
print("Delay DataFrame Shape:", delay_df.shape)

In [ ]:
#| inlcude: false
# Summary statistics for Delay Data (formatted)
delay_summary = delay_df.describe().T  # Transpose for better readability
delay_summary.style.set_caption("Delay Data Summary").format("{:.2f}")

In [ ]:
#| include: false

# Extract only the hour from the 'time' column and convert it to an integer (0-23)
delay_df['hour'] = delay_df['time'].str[:2].astype(int)

# Display result
print(delay_df[['time', 'hour']].head())


In [ ]:
#| include: false

# Display column types to verify
print(delay_df.dtypes)

#### Check pop_df

In [ ]:
#| include: false
print(pop_df.info())

#### Check toronto_gdf

In [ ]:
#| include: false
# List all column names in the shapefile
print("Columns in the shapefile:", toronto_gdf.columns.tolist())

In [ ]:
#| include: false
# Rename a column in toronto_gdf
toronto_gdf.rename(columns={"_id1": "neighbourhood_id"}, inplace=True)

# Display the updated columns
print(toronto_gdf.columns)

In [ ]:
#| include: false

# Plot the neighbourhood boundaries
toronto_gdf.plot(figsize=(10, 8), edgecolor="black", cmap="viridis")
plt.title("Toronto Neighbourhoods Map")
plt.show()


#### Check stations_df

In [ ]:
#| include: false

print(stations_df.info())

# Change 'station' column to category type
stations_df['station'] = stations_df['station'].astype('category')
print(stations_df.dtypes)

#### Check weather_df

In [ ]:
#| include: false
# Checking for missing values and duplicates in weather_df
print("\nMissing Values in Weather DataFrame:\n", weather_df.isnull().sum())
print("\nDuplicate Rows in Weather DataFrame:", weather_df.duplicated().sum())


In [ ]:
#| include: false

# check the date of these na entries in snow_depth
print(weather_df[pd.isna(weather_df['snow_depth'])].head())

In [ ]:
#| include: false

# replace NaN entries in snow_depth to 0
weather_df['snow_depth'] = weather_df['snow_depth'].fillna(0)

print("\nMissing Values in Weather DataFrame:\n", weather_df.isnull().sum())

In [ ]:
#| include: false

# Ensure 'date' column is in datetime format
weather_df['date'] = pd.to_datetime(weather_df['date'])

# Extract the hour from the 'date' column and create a new 'hour' column (numeric type)
weather_df['hour'] = weather_df['date'].dt.strftime('%H').astype(int)

# Convert 'date' column to only contain the date (drop time) and change type to datetime64[ns]
weather_df['date'] = weather_df['date'].dt.date  # Convert to date only
weather_df['date'] = pd.to_datetime(weather_df['date'])  # Convert back to datetime64[ns]

# Display result
print(weather_df.dtypes)
print(weather_df.head())

In [ ]:
#| include: false
# Summary statistics for Weather Data (formatted)
weather_summary = weather_df.describe().T  # Transpose for better readability
weather_summary.style.set_caption("Weather Data Summary").format("{:.2f}")

### Data Wrangling

In [ ]:
#| include: false
print("2024 Subway Delay data", delay_df.info())
print("2021 Neighbourhood population data", pop_df.info())
print("Neighbourhood shapefile", toronto_gdf.info())
print("Station Data", stations_df.info())
print("2024 Hourly Weather data", weather_df.info())

In [ ]:
#| include: false

# Merge population data into the Toronto GeoDataFrame
toronto_gdf = toronto_gdf.merge(pop_df, on="neighbourhood_id", how="left")

# Display result
print(toronto_gdf.head())


In [ ]:
#| include: false

# Convert stations_df to a GeoDataFrame with geometry
stations_gdf = gpd.GeoDataFrame(
    stations_df,
    geometry=gpd.points_from_xy(stations_df["longitude"], stations_df["latitude"]),
    crs="EPSG:4326"  # Ensure CRS is set correctly
)

# Convert toronto_gdf to the same CRS
toronto_gdf = toronto_gdf.to_crs(epsg=32617)  # UTM Zone 17N (for accurate distance)
stations_gdf = stations_gdf.to_crs(epsg=32617)

# Buffer stations by 0.5 km (500 meters)
stations_gdf["buffer"] = stations_gdf.geometry.buffer(500)

# Ensure the active geometry column is set **before spatial join**
stations_gdf.set_geometry("buffer", inplace=True)

# Spatial join: Find neighbourhoods within 0.5 km of each station
service_population = gpd.sjoin(toronto_gdf, stations_gdf, predicate="intersects")

# Sum the population for each station
service_population = service_population.groupby("station")["population"].sum().reset_index()

# Merge the population count back into stations_df
stations_df = stations_df.merge(service_population, on="station", how="left")

# Rename column for clarity
stations_df.rename(columns={"population": "service_population"}, inplace=True)

# Display result
print(stations_df.head())



In [ ]:
#| include: false

# Merge on date and hour
merged_df = delay_df.merge(weather_df, on=["date", "hour"], how="inner")

# Display result
print(merged_df.head())


In [ ]:
#| include: false

# Merge on station names being equal
merged_df = merged_df.merge(
    stations_df[["station", "service_population"]],  # Use "station" from stations_df
    left_on="cleaned_station",  # Use "cleaned_station" from merged_df
    right_on="station",  # Merge with "station" from stations_df
    how="left"
)

# Drop redundant "station" column from stations_df if needed
merged_df.drop(columns=["station"], inplace=True)

# Display result
print(merged_df.head())
print(merged_df.info())


In [ ]:
#| include: false
# Checking for missing values and duplicates in merged_df
print("\nMissing Values in Merged DataFrame:\n", merged_df.isnull().sum())
print("\nDuplicate Rows in Merged DataFrame:", merged_df.duplicated().sum())

### EDA

In [ ]:
#| echo: false

# Select numeric columns
numeric_cols = merged_df.select_dtypes(include=['number']).columns

# Compute summary statistics
summary_stats = merged_df[numeric_cols].describe().T  # Transpose for better readability

# Format the summary statistics professionally
styled_summary = summary_stats.style.set_caption("Summary Statistics of Merged Data").format("{:.2f}")

# Display in Jupyter Notebook
styled_summary


In [ ]:
#| echo: false

# Distribution of Delay Duration
plt.figure(figsize=(10,6))
sns.histplot(merged_df['min_delay'], bins=30, kde=True)
plt.title("Distribution of Subway Delay Duration")
plt.xlabel("Minutes Delayed")
plt.ylabel("Frequency")

# Adjust spacing so caption is at the bottom
plt.subplots_adjust(bottom=0.15)  # Moves the entire plot up

#  Add caption below the plot
plt.figtext(0.5, 0.02, "Figure: This plot shows the distribution of subway delay durations.", 
            ha="center", fontsize=10, wrap=True)
plt.show()


In [ ]:
#| echo: false

# Get the top 5 most frequent delay codes
top_delay_codes = (
    merged_df['code']
    .value_counts()
    .nlargest(5)  # Select top 5
    .reset_index()
)
top_delay_codes.columns = ['code', 'count']

# Filter the original dataframe to only include the top 5 codes
filtered_df = merged_df[merged_df['code'].isin(top_delay_codes['code'])]

# Plot
plt.figure(figsize=(10, 6))
sns.countplot(data=filtered_df, y='code', order=top_delay_codes['code'], palette='viridis')

plt.title("Top 5 Most Frequent Delay Codes")
plt.xlabel("Number of Delays")
plt.ylabel("Delay Code")

# Adjust spacing so caption is at the bottom
plt.subplots_adjust(bottom=0.15)  # Moves the entire plot up

#  Add caption below the plot
plt.figtext(0.5, 0.02, "Figure: This plot shows the most frequent delay codes.", 
            ha="center", fontsize=10, wrap=True)

plt.show()

In [ ]:
#| include: false
# SUDP = Disorderly Patron
# MUPAA = Passenger Assistance Alarm Activated - No Trouble Found
# SUO = Passenger Other
# PUOPO = OPTO (COMMS) Train Door Monitoring
# MUIR = Injured or ill Customer (On Train) - Medical Aid Refused

In [ ]:
#| echo: false
# Delays by Line
plt.figure(figsize=(12,6))
sns.boxplot(x=merged_df['line'], y=merged_df['min_delay'])
plt.xticks()
plt.title("Distribution of Delays Across Subway Lines")
plt.xlabel("Subway Line")
plt.ylabel("Minutes Delayed")

# Adjust spacing so caption is at the bottom
plt.subplots_adjust(bottom=0.15)  # Moves the entire plot up

#  Add caption below the plot
plt.figtext(0.5, 0.02, "Figure: This plot shows the distribution of delays accross subway lines.", 
            ha="center", fontsize=10, wrap=True)

plt.show()


In [ ]:
#| echo: false

# make 'cleaned_station' str type (will reverse at the end)
merged_df["cleaned_station"] = merged_df["cleaned_station"].astype(str)

# Get the top 3 stations per line based on delay count (ensuring no duplicates across lines)
top_stations_per_line = (
    merged_df.groupby(['line', 'cleaned_station'])
    .size()
    .reset_index(name="delay_count")
    .sort_values(['line', 'delay_count'], ascending=[True, False])  # Sort within each line
    .groupby('line')  # Group by line
    .head(3)  # Select top 3 stations per line
)

# Filter delay_df to include only the selected top stations
filtered_df = merged_df.merge(top_stations_per_line[['line', 'cleaned_station']], on=['line', 'cleaned_station'])

# Compute average delay for these stations per line
avg_delay_per_line = (
    filtered_df.groupby(['line', 'cleaned_station'])['min_delay']
    .mean()
    .reset_index()
)

# Create Faceted Plot by Line with stations on X-axis and shared Y-axis
g = sns.catplot(
    data=avg_delay_per_line,
    x="cleaned_station",  # Stations on X-axis
    y="min_delay",  # Average delay time on Y-axis
    col="line",  # Facet by subway line
    kind="bar",
    col_wrap=1,  # Adjust layout
    height=7,  # Make plot taller
    aspect=2,  # Widen subplots
    palette="viridis",
    sharey=True,  # Use shared Y-axis for uniformity
    sharex=False,  # Ensure correct spacing of station names
    legend=False  # Remove legend
)

g.set_axis_labels("Station", "Average Delay (Minutes)")
g.set_titles("{col_name} Line")
g.fig.suptitle("Average Delay Time for the Top 3 Stations per Line", y=1.05, fontsize=16)
    
# Adjust spacing so caption is at the bottom
plt.subplots_adjust(bottom=0.06)  # Moves the entire plot up
    
#  Add caption below the plot
plt.figtext(0.5, 0.02, "Figure: This plot shows the average delay time for the top 3 most delayed stations per subway line.", 
            ha="center", fontsize=10, wrap=True)

plt.show()

# return 'cleaned_station' to categorical type
merged_df["cleaned_station"] = merged_df["cleaned_station"].astype('category')

In [ ]:
#| echo: false

# Delays by Bound
plt.figure(figsize=(12,6))
sns.boxplot(x=merged_df['bound'], y=merged_df['min_delay'])
plt.xticks()
plt.title("Distribution of Delays Across Subway Bounds")
plt.xlabel("Bound")
plt.ylabel("Minutes Delayed")

# Adjust spacing so caption is at the bottom
plt.subplots_adjust(bottom=0.15)  # Moves the entire plot up

#  Add caption below the plot
plt.figtext(0.5, 0.02, "Figure: This plot shows the distribution of delays accross subway bounds.", 
            ha="center", fontsize=10, wrap=True)

plt.show()

In [ ]:
#| include: false

# Aggregate delay count and average delay per hour
delay_by_hour = merged_df.groupby('hour').agg(
    delay_count=('min_delay', 'count'),
    avg_delay=('min_delay', 'mean')
).reset_index()

# Frequency of Delays per Hour
plt.figure(figsize=(12, 6))
sns.barplot(x=delay_by_hour['hour'], y=delay_by_hour['delay_count'], color='blue', alpha=0.7)
plt.title("Subway Delay Frequency by Hour")
plt.xlabel("Hour of the Day")
plt.ylabel("Number of Delays")
plt.xticks(range(0, 24))  # Ensure x-axis covers all hours (0-23)

# Adjust spacing so caption is at the bottom
plt.subplots_adjust(bottom=0.15)  # Moves the entire plot up

#  Add caption below the plot
plt.figtext(0.5, 0.02, "Figure: This plot shows the frequency of delays by hour.", 
            ha="center", fontsize=10, wrap=True)

plt.show()

In [ ]:
#| echo: false

# Average Delay Duration per Hour
plt.figure(figsize=(12, 6))
sns.lineplot(x=delay_by_hour['hour'], y=delay_by_hour['avg_delay'], marker='o', color='red')
plt.title("Average Delay Duration by Hour")
plt.xlabel("Hour of the Day")
plt.ylabel("Average Delay (Minutes)")
plt.xticks(range(0, 24))  # Ensure x-axis covers all hours (0-23)
plt.grid(True)  # Add grid for readability

# Adjust spacing so caption is at the bottom
plt.subplots_adjust(bottom=0.15)  # Moves the entire plot up

#  Add caption below the plot
plt.figtext(0.5, 0.02, "Figure: This plot shows the delay duration by hour.", 
            ha="center", fontsize=10, wrap=True)

plt.show()

In [ ]:
#| echo: false

# Plot
plt.figure(figsize=(10, 6))
sns.boxplot(x=merged_df['day'], y=merged_df['min_delay'], order=day_order, palette="coolwarm")
plt.xticks()
plt.title("Distribution of Delays Across Days")
plt.xlabel("Day of the Week")
plt.ylabel("Minutes Delayed")

# Adjust spacing so caption is at the bottom
plt.subplots_adjust(bottom=0.15)  # Moves the entire plot up

#  Add caption below the plot
plt.figtext(0.5, 0.02, "Figure: This plot shows the distribution of delays accross days of the week.", 
            ha="center", fontsize=10, wrap=True)

plt.show()

In [ ]:
#| echo: false
plt.figure(figsize=(10, 6))
sns.regplot(x="service_population", y="min_delay", data=merged_df, scatter_kws={"alpha": 0.5}, line_kws={"color": "red"})
plt.title("Service Population On Delays", fontsize=14)
plt.xlabel("Service Population", fontsize=12)
plt.ylabel("Delay (Minutes)", fontsize=12)
plt.grid(True)

# Adjust spacing so caption is at the bottom
plt.subplots_adjust(bottom=0.15)  # Moves the entire plot up

#  Add caption below the plot
plt.figtext(0.5, 0.02, "Figure: This plot shows the impact of service population size on the duration of a delay.", 
            ha="center", fontsize=10, wrap=True)

plt.show()


In [ ]:
#| echo: false

# Compute correlation matrix 2
corr2_matrix = merged_df[["min_delay", "temperature", "apparent_temperature", "relative_humidity", 
                          "precipitation", "rain", "snowfall", "snow_depth", "cloud_cover", "wind_speed", 
                          "wind_gusts", "hour", "service_population"]].corr()

# Plot heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr2_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Weather & Delays")
plt.show()


In [ ]:
#| echo: false
# Fit the ANOVA model
anova_model = smf.ols("min_delay ~ C(day) + C(bound) + C(line) + C(cleaned_station)", data=merged_df).fit()

# Perform ANOVA test
anova_table = sm.stats.anova_lm(anova_model)

# Format table for better display
styled_anova = anova_table.style.set_caption("ANOVA Results: Impact of Subway Line on Delay Time") \
                               .format("{:.4f}") \
                               .set_table_styles([
                                   {'selector': 'thead', 'props': [('font-weight', 'bold'), ('background-color', '#f4f4f4')]},
                                   {'selector': 'tbody tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]}
                               ])

# Display the formatted table
styled_anova